# Sesión 2 — Búsqueda heurística

In [1]:
import random
import heapq
import math
import time
import numpy as np

from collections import deque, Counter
from itertools import combinations
from fractions import Fraction
import networkx as nx

class Problem(object):
    """Clase abstracta para un problema de búsqueda (AIMA-Python)."""
    def __init__(self, initial=None, goal=None, **kwds):
        self.__dict__.update(initial=initial, goal=goal, **kwds)

    def actions(self, state):
        raise NotImplementedError

    def result(self, state, action):
        raise NotImplementedError

    def is_goal(self, state):
        return state == self.goal

    def action_cost(self, s, a, s1):
        return 1

    def h(self, node):
        return 0

    def __str__(self):
        return '{}({!r}, {!r})'.format(type(self).__name__, self.initial, self.goal)


class Node:
    """Nodo de un árbol de búsqueda."""
    def __init__(self, state, parent=None, action=None, path_cost=0):
        self.__dict__.update(
            state=state,
            parent=parent,
            action=action,
            path_cost=path_cost
        )

    def __repr__(self):
        return '<{}>'.format(self.state)

    def __len__(self):
        return 0 if self.parent is None else 1 + len(self.parent)

    def __lt__(self, other):
        return self.path_cost < other.path_cost


failure = Node('failure', path_cost=math.inf)
cutoff = Node('cutoff', path_cost=math.inf)


def expand(problem, node):
    s = node.state
    for action in problem.actions(s):
        s1 = problem.result(s, action)
        cost = node.path_cost + problem.action_cost(s, action, s1)
        yield Node(s1, node, action, cost)


def path_actions(node):
    if node.parent is None:
        return []
    return path_actions(node.parent) + [node.action]


def path_states(node):
    if node in (cutoff, failure, None):
        return []
    return path_states(node.parent) + [node.state]


class PriorityQueue:
    """Cola que extrae primero el elemento con menor valor de key."""
    def __init__(self, items=(), key=lambda x: x):
        self.key = key
        self.items = []
        for item in items:
            self.add(item)

    def add(self, item):
        heapq.heappush(self.items, (self.key(item), item))

    def pop(self):
        return heapq.heappop(self.items)[1]

    def top(self):
        return self.items[0][1]

    def f_top(self):
        return self.items[0][0]

    def __len__(self):
        return len(self.items)


In [ ]:
NODOS_EXPANDIDOS = 0
MAXIMO_FRONTIER = 0
NODOS_DUMMY = 0
ACTUALIZACIONES_EN_REACHED = 0
ESTADOS_REEXPANDIDOS = 0
HIJOS_DESCARTADOS = 0
ESTADOS_ALCANZADOS = 0
FINAL_FRONTIER = 0

def best_first_search(problem, f):
    """Best-First Search con descarte de entradas obsoletas de frontier."""
    global NODOS_EXPANDIDOS, MAXIMO_FRONTIER, NODOS_DUMMY, ACTUALIZACIONES_EN_REACHED, ESTADOS_REEXPANDIDOS, HIJOS_DESCARTADOS, ESTADOS_ALCANZADOS, FINAL_FRONTIER

    node = Node(problem.initial)
    frontier = PriorityQueue([node], key=f)
    reached = {problem.initial: node}
    expanded = {}

    NODOS_EXPANDIDOS = 0
    ESTADOS_REEXPANDIDOS = 0
    MAXIMO_FRONTIER = 0
    NODOS_DUMMY = 0
    ACTUALIZACIONES_EN_REACHED = 0
    HIJOS_DESCARTADOS = 0
    ESTADOS_ALCANZADOS = 0
    FINAL_FRONTIER = 0

    while frontier:
        f_node = frontier.f_top()
        node = frontier.pop()

        # Nodo obsoleto: posteriormente se encontró un camino mejor
        # hacia el mismo estado.
        if node.path_cost > reached[node.state].path_cost:
            NODOS_DUMMY += 1
            continue

        if (node.state in expanded):
            ESTADOS_REEXPANDIDOS += 1
        else:
            expanded[node.state] = True 

        if problem.is_goal(node.state):
            ESTADOS_ALCANZADOS = len(reached)
            FINAL_FRONTIER = len(frontier)
            return node

        NODOS_EXPANDIDOS += 1
        print("f_node: ", f_node, "h_node: ", f_node - node.path_cost)

        for child in expand(problem, node):
            s = child.state

            if s not in reached or child.path_cost < reached[s].path_cost:
                if s in reached:
                    ACTUALIZACIONES_EN_REACHED += 1
                else:
                    HIJOS_DESCARTADOS += 1
                reached[s] = child
                frontier.add(child)

        MAXIMO_FRONTIER = max(MAXIMO_FRONTIER, len(frontier))

    return failure


def g(n):
    return n.path_cost


def astar(problem, h=None):
    h = h or problem.h
    return best_first_search(problem, f=lambda n: g(n) + h(n))


def weighted_astar(problem, h=None, weight=1.4):
    h = h or problem.h
    return best_first_search(problem, f=lambda n: g(n) + weight * h(n))


def pea(problem, h=None, epsilon=0.5):
    return weighted_astar(problem, h=h, weight=1 + epsilon)


def greedy(problem, h=None):
    h = h or problem.h
    return best_first_search(problem, f=h)


def uniform_cost(problem):
    return best_first_search(problem, f=g)


## 8-puzzle

In [3]:
class EightPuzzle(Problem):
    def __init__(self, initial, goal=(1, 2, 3, 8, 0, 4, 7, 6, 5)):
        assert inversions(initial) % 2 == inversions(goal) % 2, \
            "El estado inicial y el objetivo no son alcanzables."
        self.initial, self.goal = tuple(initial), tuple(goal)

    def actions(self, state):
        moves = ((1, 3), (0, 2, 4), (1, 5),
                 (0, 4, 6), (1, 3, 5, 7), (2, 4, 8),
                 (3, 7), (4, 6, 8), (7, 5))
        return moves[state.index(0)]

    def result(self, state, action):
        s = list(state)
        blank = state.index(0)
        s[action], s[blank] = s[blank], s[action]
        return tuple(s)

    def h0(self, node):
        return 0

    def h1(self, node):
        """Número de fichas mal colocadas, excluyendo el hueco."""
        return sum(a != b and a != 0
                   for a, b in zip(node.state, self.goal))

    def h2(self, node):
        """Suma de distancias Manhattan respecto al objetivo."""
        total = 0
        for tile in range(1, 9):
            p = node.state.index(tile)
            q = self.goal.index(tile)
            total += abs((p % 3) - (q % 3)) + abs((p // 3) - (q // 3))
        return total

    def h3(self, node):
        """
        Admisible pero no consistente.

        Solo contabiliza las fichas cuya distancia Manhattan al objetivo
        es exactamente 2, asignando una contribución 2.

        Es admisible porque h3(n) <= h2(n).

        No es consistente porque una ficha puede pasar en un movimiento
        de distancia 2 a distancia 1. Su contribución pasa de 2 a 0 y
        puede ocurrir h3(n) > 1 + h3(n').
        """
        total = 0
        for tile in range(1, 9):
            p = node.state.index(tile)
            q = self.goal.index(tile)
            d = abs((p % 3) - (q % 3)) + abs((p // 3) - (q // 3))
            if d == 2:
                total += 2
        return total

    def h4(self, node):
        """
        No admisible.

        h4(n) = 2*h2(n). Puede sobreestimar el coste real restante.
        Por ejemplo, a un movimiento del objetivo h2(n)=1 pero h4(n)=2.
        """
        return 2 * self.h2(node)

    def h(self, node):
        return self.h1(node)


def inversions(board):
    return sum((a > b and a != 0 and b != 0)
               for a, b in combinations(board, 2))


def board8(board, fmt=(3 * '{} {} {}\n')):
    return fmt.format(*board).replace('0', '_')


In [4]:
ejemplos_8puzzle = {
 5: [(1,0,3,8,2,5,7,4,6),(1,4,2,0,8,3,7,6,5),(1,3,4,8,6,2,7,0,5),(1,2,3,7,8,0,6,5,4)],
10: [(8,2,1,7,0,3,6,5,4),(1,4,0,8,5,2,7,3,6),(8,1,3,7,0,5,4,2,6),(8,1,2,4,0,6,7,5,3)],
15: [(4,8,2,6,3,5,1,0,7),(1,4,5,2,7,0,8,6,3),(1,3,8,6,7,4,2,0,5),(2,0,8,7,5,3,4,1,6)],
20: [(6,2,7,4,5,1,0,8,3),(4,7,2,1,0,6,3,5,8),(7,1,5,4,0,8,2,6,3),(5,1,6,4,0,3,8,7,2)],
25: [(6,7,4,0,5,1,3,2,8),(6,0,7,5,4,1,3,8,2),(3,4,8,5,7,1,6,0,2)],
30: [(5,6,7,2,8,4,0,3,1),(5,6,7,4,0,8,3,2,1),(5,4,7,6,0,3,8,2,1)]
}

problemas_8puzzle = {
    d: [EightPuzzle(s) for s in estados]
    for d, estados in ejemplos_8puzzle.items()
}


In [ ]:
p = problemas_8puzzle[30][0]
print(p)
t0 = time.perf_counter()
sol = astar(p, h=p.h1)
t = time.perf_counter() - t0

acciones = path_actions(sol)
#recorrido = [0] + [a for a in acciones if a != -1] + [0]

print("Solucion:", acciones)
print("Coste:", sol.path_cost)
print("Tiempo:", t)
print("Nodos expandidos:", NODOS_EXPANDIDOS)
print("Estados reexpandidos:", ESTADOS_REEXPANDIDOS)
print("Estados alcanzados:", ESTADOS_ALCANZADOS)
print("Máximo frontier:", MAXIMO_FRONTIER)
print("Final frontier:", FINAL_FRONTIER)
print("Nodos dummy:", NODOS_DUMMY)
print("Actualizaciones en reached:", ACTUALIZACIONES_EN_REACHED)
print("Hijos descartados:", HIJOS_DESCARTADOS)


## TSP


In [ ]:
class EstadoTSP:
    def __init__(self, actual, visitadas):
        self.actual = actual
        self.visitadas = tuple(sorted(visitadas))

    def __hash__(self):
        return hash((self.actual, self.visitadas))

    def __eq__(self, other):
        return (
            isinstance(other, EstadoTSP)
            and self.actual == other.actual
            and self.visitadas == other.visitadas
        )

    def __repr__(self):
        return f'[Visitadas: {list(self.visitadas)} Actual: {self.actual}]'

class Grafo:
    def __init__(self, lista):
        self.N = int((((8 * len(lista) + 1) ** 0.5) - 1) / 2)
        self.ciudades = list(range(self.N))
        self.Dist = [[0] * self.N for _ in self.ciudades]

        acc = 0
        for x in range(self.N):
            acc += x
            for y in range(x + 1):
                self.Dist[x][y] = self.Dist[y][x] = lista[acc + y]

def hungarian_min_cost(cost):
    """Coste mínimo de una asignación cuadrada."""
    n = len(cost)
    if n == 0:
        return 0.0

    m = len(cost[0])
    u = [0.0] * (n + 1)
    v = [0.0] * (m + 1)
    p = [0] * (m + 1)
    way = [0] * (m + 1)

    for i in range(1, n + 1):
        p[0] = i
        j0 = 0
        minv = [float('inf')] * (m + 1)
        used = [False] * (m + 1)

        while True:
            used[j0] = True
            i0 = p[j0]
            delta = float('inf')
            j1 = 0

            for j in range(1, m + 1):
                if not used[j]:
                    cur = cost[i0 - 1][j - 1] - u[i0] - v[j]

                    if cur < minv[j]:
                        minv[j] = cur
                        way[j] = j0

                    if minv[j] < delta:
                        delta = minv[j]
                        j1 = j

            for j in range(m + 1):
                if used[j]:
                    u[p[j]] += delta
                    v[j] -= delta
                else:
                    minv[j] -= delta

            j0 = j1

            if p[j0] == 0:
                break

        while True:
            j1 = way[j0]
            p[j0] = p[j1]
            j0 = j1

            if j0 == 0:
                break

    assignment = [-1] * n

    for j in range(1, m + 1):
        if p[j] != 0:
            assignment[p[j] - 1] = j - 1

    return sum(cost[i][assignment[i]] for i in range(n))


def _fraction_to_number(value):
    """Convierte una Fraction exacta a int cuando es posible."""
    if isinstance(value, Fraction):
        if value.denominator == 1:
            return int(value.numerator)
        return float(value)
    return value


def minimum_weight_edge_cover_edmonds(grafo, nodes):
    """
    h2: Minimum Weight Edge Cover sobre un grafo NO dirigido.

    Se resuelve exactamente mediante la reducción a Minimum Weight
    Perfect Matching y un algoritmo de matching de Edmonds.

    Construcción:
      1. Se crean dos copias del grafo residual.
      2. Cada arista original aparece en ambas copias con su mismo coste.
      3. Cada vértice v se conecta con su copia mediante una arista de
         coste 2 * mu(v), donde mu(v) es la arista más barata incidente en v.
      4. Un perfect matching mínimo en el grafo auxiliar tiene coste igual
         al doble del Minimum Weight Edge Cover original.
    """
    nodes = list(dict.fromkeys(nodes))
    n = len(nodes)

    if n <= 1:
        return 0

    mu = {
        u: min(
            grafo.Dist[u][v]
            for v in nodes
            if v != u
        )
        for u in nodes
    }

    auxiliar = nx.Graph()

    for u in nodes:
        auxiliar.add_node(("L", u))
        auxiliar.add_node(("R", u))

        auxiliar.add_edge(
            ("L", u),
            ("R", u),
            weight=2 * mu[u]
        )

    for u, v in combinations(nodes, 2):
        coste = grafo.Dist[u][v]

        auxiliar.add_edge(
            ("L", u),
            ("L", v),
            weight=coste
        )

        auxiliar.add_edge(
            ("R", u),
            ("R", v),
            weight=coste
        )

    matching = nx.min_weight_matching(
        auxiliar,
        weight="weight"
    )

    # El grafo auxiliar siempre admite un perfect matching mediante
    # las aristas que unen cada vértice con su copia.
    if len(matching) != n:
        raise RuntimeError(
            "No se ha podido obtener el perfect matching del grafo auxiliar."
        )

    coste_doble = sum(
        auxiliar[u][v]["weight"]
        for u, v in matching
    )

    return _fraction_to_number(
        Fraction(str(coste_doble)) / 2
    )


def _scale_fractions(values):
    """
    Escala una colección de Fraction a enteros sin perder exactitud.

    Se utiliza para introducir un segundo criterio de desempate en
    el matching sin modificar el criterio principal.
    """
    values = list(values)

    if not values:
        return [], 1

    denominator = 1

    for value in values:
        denominator = math.lcm(
            denominator,
            value.denominator
        )

    return [
        int(value * denominator)
        for value in values
    ], denominator


def _partial_nonnegative_cover_extreme(
    nodes,
    edges,
    shifted,
    required,
    prefer_more_matching
):
    """
    Resuelve el problema de cubrir solo los vértices de 'required'
    cuando todos los costes incidentes en ellos son no negativos.

    Se usa la reducción de Minimum Weight Edge Cover a Maximum Weight
    Matching. El segundo criterio de desempate permite obtener, entre
    todos los covers de coste mínimo, el de menor o mayor cardinalidad.

    Devuelve:
        coste mínimo adicional,
        cardinalidad del cover adicional.
    """
    required = set(required)

    if not required:
        return Fraction(0), 0

    mu = {}

    for v in required:
        incident = [
            shifted[e]
            for e in edges
            if v in e
        ]

        if not incident:
            raise RuntimeError(
                "Existe un vértice aislado en el problema de cobertura."
            )

        mu[v] = min(incident)

        if mu[v] < 0:
            raise RuntimeError(
                "La rutina parcial solo admite costes no negativos."
            )

    matching_edges = []
    gains = []

    for e in edges:
        u, v = e

        if u in required and v in required:
            matching_edges.append(e)
            gains.append(
                mu[u] + mu[v] - shifted[e]
            )

    integer_gains, _ = _scale_fractions(gains)

    matching_graph = nx.Graph()
    matching_graph.add_nodes_from(required)

    # Una diferencia de una unidad en el criterio principal debe pesar
    # más que cualquier posible diferencia en el número de aristas.
    secondary_scale = len(required) // 2 + 1

    gain_by_edge = {}

    for e, gain, integer_gain in zip(
        matching_edges,
        gains,
        integer_gains
    ):
        secondary = 1 if prefer_more_matching else -1

        matching_graph.add_edge(
            e[0],
            e[1],
            weight=(
                integer_gain * secondary_scale
                + secondary
            )
        )

        gain_by_edge[frozenset(e)] = gain

    matching = nx.max_weight_matching(
        matching_graph,
        maxcardinality=False,
        weight="weight"
    )

    total_gain = sum(
        (
            gain_by_edge[frozenset(e)]
            for e in matching
        ),
        Fraction(0)
    )

    cost = (
        sum(mu.values(), Fraction(0))
        - total_gain
    )

    # En la construcción canónica, cada arista del matching cubre
    # dos vértices requeridos y cada vértice restante necesita una
    # arista adicional.
    cardinality = (
        len(required)
        - len(matching)
    )

    return cost, cardinality


def _minimum_shifted_cover_extremes(
    nodes,
    costs,
    lamb
):
    """
    Para los costes desplazados c_e - lambda calcula:

        phi(lambda) = min_C sum_{e in C} (c_e - lambda)

    junto con la menor y la mayor cardinalidad entre todos los covers
    que alcanzan ese mismo valor óptimo.

    Esta es la operación básica del método paramétrico de White y Gillenson.
    """
    edges = list(costs)

    shifted = {
        e: costs[e] - lamb
        for e in edges
    }

    # Menor cardinalidad entre los covers óptimos.
    #
    # Toda arista de coste estrictamente negativo debe pertenecer a
    # cualquier solución óptima. Las aristas de coste cero se incorporan
    # solo cuando son necesarias o ayudan a reducir la cardinalidad.
    negative_edges = {
        e
        for e in edges
        if shifted[e] < 0
    }

    covered_by_negative = {
        v
        for e in negative_edges
        for v in e
    }

    required = (
        set(nodes)
        - covered_by_negative
    )

    additional_cost, additional_cardinality = (
        _partial_nonnegative_cover_extreme(
            nodes,
            edges,
            shifted,
            required,
            prefer_more_matching=True
        )
    )

    phi_min = (
        sum(
            (
                shifted[e]
                for e in negative_edges
            ),
            Fraction(0)
        )
        + additional_cost
    )

    k_min = (
        len(negative_edges)
        + additional_cardinality
    )

    # Mayor cardinalidad entre los covers óptimos.
    #
    # Todas las aristas de coste cero pueden añadirse sin cambiar
    # el valor de la función objetivo.
    nonpositive_edges = {
        e
        for e in edges
        if shifted[e] <= 0
    }

    covered_by_nonpositive = {
        v
        for e in nonpositive_edges
        for v in e
    }

    required = (
        set(nodes)
        - covered_by_nonpositive
    )

    additional_cost, additional_cardinality = (
        _partial_nonnegative_cover_extreme(
            nodes,
            edges,
            shifted,
            required,
            prefer_more_matching=False
        )
    )

    phi_max = (
        sum(
            (
                shifted[e]
                for e in negative_edges
            ),
            Fraction(0)
        )
        + additional_cost
    )

    k_max = (
        len(nonpositive_edges)
        + additional_cardinality
    )

    if phi_min != phi_max:
        raise RuntimeError(
            "Error interno en el cálculo paramétrico del q-cover."
        )

    return phi_min, k_min, k_max


def minimum_q_cover_white_gillenson(
    grafo,
    nodes,
    q
):
    """
    h3: Minimum q-Cover de White y Gillenson.

    Problema:
        seleccionar exactamente q aristas NO dirigidas distintas,
        cubrir todos los vértices residuales y minimizar el coste.

    Se implementa el método paramétrico de White y Gillenson:

        c_e(lambda) = c_e - lambda.

    Para un lambda dado se resuelve exactamente el Minimum Weight
    Edge Cover con los costes desplazados. Si entre los covers óptimos
    existe uno de cardinalidad q, el lema paramétrico garantiza que
    ese cover es también un Minimum q-Cover para los costes originales.

    La búsqueda de lambda utiliza las intersecciones de las rectas
    asociadas a covers óptimos de distinta cardinalidad. Se trabaja
    con Fraction para evitar errores numéricos en los puntos de cambio.
    """
    nodes = list(dict.fromkeys(nodes))
    n = len(nodes)

    if n <= 1:
        if q == 0:
            return 0

        raise ValueError(
            "No existe un q-cover con la cardinalidad indicada."
        )

    edges = list(combinations(nodes, 2))

    costs = {
        e: Fraction(
            str(grafo.Dist[e[0]][e[1]])
        )
        for e in edges
    }

    m = len(edges)

    # En un grafo completo sin vértices aislados, la cardinalidad mínima
    # de un edge cover es ceil(n / 2).
    minimum_cardinality = (n + 1) // 2

    if q < minimum_cardinality or q > m:
        raise ValueError(
            f"No existe un {q}-cover para {n} vértices."
        )

    # Para lambda suficientemente pequeño se penaliza fuertemente usar
    # muchas aristas. Para lambda mayor que cualquier coste, todas las
    # aristas tienen coste desplazado negativo y se seleccionan todas.
    span = (
        sum(
            (
                abs(cost)
                for cost in costs.values()
            ),
            Fraction(0)
        )
        + 1
    )

    lambda_left = -span
    lambda_right = max(costs.values()) + 1

    def evaluate(lamb):
        return _minimum_shifted_cover_extremes(
            nodes,
            costs,
            lamb
        )

    phi_left, kmin_left, kmax_left = evaluate(
        lambda_left
    )

    if kmin_left <= q <= kmax_left:
        return _fraction_to_number(
            phi_left + lambda_left * q
        )

    phi_right, kmin_right, kmax_right = evaluate(
        lambda_right
    )

    if kmin_right <= q <= kmax_right:
        return _fraction_to_number(
            phi_right + lambda_right * q
        )

    if not (
        kmax_left < q
        and q < kmin_right
    ):
        raise RuntimeError(
            "No se ha podido acotar el parámetro lambda."
        )

    # El número de cardinalidades posibles es finito. Cada iteración
    # introduce un nuevo punto de cambio de la función paramétrica.
    max_iterations = m + 5

    for _ in range(max_iterations):
        k_left = kmax_left
        original_left = (
            phi_left
            + lambda_left * k_left
        )

        k_right = kmin_right
        original_right = (
            phi_right
            + lambda_right * k_right
        )

        # Valor de lambda en el que las dos soluciones consideradas
        # tienen exactamente el mismo coste desplazado.
        lamb = (
            (original_right - original_left)
            / (k_right - k_left)
        )

        phi, k_min, k_max = evaluate(lamb)

        # Por convexidad de los costes mínimos respecto a la cardinalidad,
        # si q está entre las cardinalidades mínima y máxima de los covers
        # óptimos para este lambda, existe un cover óptimo de cardinalidad q.
        if k_min <= q <= k_max:
            return _fraction_to_number(
                phi + lamb * q
            )

        if k_max < q:
            if lamb == lambda_left:
                raise RuntimeError(
                    "La búsqueda paramétrica no progresa."
                )

            lambda_left = lamb
            phi_left = phi
            kmin_left = k_min
            kmax_left = k_max

        else:
            if lamb == lambda_right:
                raise RuntimeError(
                    "La búsqueda paramétrica no progresa."
                )

            lambda_right = lamb
            phi_right = phi
            kmin_right = k_min
            kmax_right = k_max

    raise RuntimeError(
        "No se ha encontrado el Minimum q-Cover."
    )


class SymmetricTSP(Problem):
    def __init__(self, grafo):
        self.grafo = grafo
        self.initial = EstadoTSP(0, ())

    def actions(self, state):
        if state.actual == -1:
            return []

        no_visitadas = [
            c for c in self.grafo.ciudades
            if c != state.actual and c not in state.visitadas
        ]

        return no_visitadas if no_visitadas else [-1]

    def result(self, state, action):
        visitadas = set(state.visitadas)
        visitadas.add(state.actual)
        return EstadoTSP(action, visitadas)

    def is_goal(self, state):
        return state.actual == -1

    def action_cost(self, s, a, s1):
        return (
            self.grafo.Dist[s.actual][0]
            if a == -1
            else self.grafo.Dist[s.actual][a]
        )

    def _unvisited(self, state):
        return [
            c for c in self.grafo.ciudades
            if c != state.actual and c not in state.visitadas
        ]

    def _residual_nodes(self, state):
        if state.actual == -1:
            return []
        return list(dict.fromkeys(
            [state.actual] + self._unvisited(state) + [0]
        ))

    def h0(self, node):
        return 0

    def hPOBRE(self, node):
        if node.state.actual == -1:
            return 0
        return len(self._unvisited(node.state)) + 1

    def h1(self, node): # q = N-k+1
        """
        h1: seleccionar las q aristas distintas más baratas
        del grafo residual NO dirigido.

        Se conserva R1. q = N-k+1
        Se relajan R2, R3 y R4.
        """
        s = node.state

        if s.actual == -1:
            return 0

        U = self._unvisited(s)

        if not U:
            return self.grafo.Dist[s.actual][0]

        nodes = self._residual_nodes(s)
        q = len(U) + 1

        edge_costs = sorted(
            self.grafo.Dist[u][v]
            for u, v in combinations(nodes, 2)
        )

        if q > len(edge_costs):
            raise RuntimeError(
                "No hay suficientes aristas distintas en el grafo residual."
            )

        return sum(edge_costs[:q])


    def h2(self, node):
        """
        h2: Minimum Weight Edge Cover.

        Se conserva R2. Que toque a todos los arcos (recubrimiento mínimo)
        Se relajan R1, R3 y R4.

        El problema relajado se resuelve de manera óptima mediante
        la reducción a Minimum Weight Perfect Matching y Edmonds.
        """
        s = node.state

        if s.actual == -1:
            return 0

        U = self._unvisited(s)

        if not U:
            return self.grafo.Dist[s.actual][0]

        nodes = self._residual_nodes(s)

        return minimum_weight_edge_cover_edmonds(
            self.grafo,
            nodes
        )

    def h3(self, node):
        """
        h3: Minimum q-Cover. q = N-k+1

        Se conservan R1 y R2.
        Se relajan R3 y R4.

        El problema relajado se resuelve de manera óptima mediante
        el método paramétrico de White y Gillenson.
        """
        s = node.state

        if s.actual == -1:
            return 0

        U = self._unvisited(s)

        if not U:
            return self.grafo.Dist[s.actual][0]

        nodes = self._residual_nodes(s)
        q = len(U) + 1

        return minimum_q_cover_white_gillenson(
            self.grafo,
            nodes,
            q
        )

    def hMST(self, node):
        """
        h4: Árbol de expansión mínimo sobre el grafo residual.

        Se conservan R1, R2 y R4.
        Se relaja R3: no se imponen los grados exactos del tour.
        """

        s = node.state

        if s.actual == -1:
            return 0

        nodes = self._residual_nodes(s)

        if len(nodes) <= 1:
            return 0

        in_tree = {nodes[0]}
        total = 0

        while len(in_tree) < len(nodes):
            best = None

            for u in in_tree:
                for v in nodes:
                    if v in in_tree or u == v:
                        continue

                    c = self.grafo.Dist[u][v]

                    if best is None or c < best[0]:
                        best = (c, v)

            total += best[0]
            in_tree.add(best[1])

        return total


    def hHUNGARO(self, node):
        """
        h5: Relajación mediante un problema de asignación.

        Se conservan R1, R2 y R3.
        Se relaja R4: se mantienen las restricciones de entrada
        y salida de cada ciudad, pero se permite formar subtours.
        """
        "El algoritmo hungaro produce una cota inferior del problama relajado"
        "dado que puede considerar varias veces el mismo arco"

        s = node.state

        if s.actual == -1:
            return 0

        U = self._unvisited(s)

        if not U:
            return self.grafo.Dist[s.actual][0]

        sources = [s.actual] + U
        dests = U + [0]

        BIG = 10**12

        matrix = [
            [
                BIG if u == v else self.grafo.Dist[u][v]
                for v in dests
            ]
            for u in sources
        ]

        return hungarian_min_cost(matrix)

    def hNN(self, node):  # >= h*(n): cota superior, NO admisible
        """Completa el recorrido mediante Nearest Neighbour.
           Desde la ciudad actual se va a la ciudad no visitada más cercana,
           desde esa a la siguiente no visitada más cercana, y así hasta
           visitarlas todas y volver a la ciudad inicial (0).
        """
        s = node.state
        if s.actual == -1:
            return 0

        pendientes = set(self._unvisited(s))
        actual = s.actual
        total = 0

        while pendientes:
            sig = min(pendientes, key=lambda c: self.grafo.Dist[actual][c])
            total += self.grafo.Dist[actual][sig]
            pendientes.remove(sig)
            actual = sig

        return total + self.grafo.Dist[actual][0]  # volver al origen

    def h(self, node):
        return self.hMST(node)


In [ ]:
ejemploClase = [
    0,
    21,0,
    12,7,0,
    15,32,5,0,
    113,25,18,180,0,
    92,9,20,39,17,0
]

gr21 = [
    0, 
    510, 0, 
    635, 355, 0, 
    91, 415, 605, 0, 
    385, 585, 390, 350, 0, 
    155, 475, 495, 120, 240, 0, 
    110, 480, 570, 78, 320, 96, 0, 
    130, 500, 540, 97, 285, 36, 29, 0, 
    490, 605, 295, 460, 120, 350, 425, 390, 0, 
    370, 320, 700, 280, 590, 365, 350, 370, 625, 0, 
    155, 380, 640, 63, 430, 200, 160, 175, 535, 240, 0, 
    68, 440, 575, 27, 320, 91, 48, 67, 430, 300, 90, 0, 
    610, 360, 705, 520, 835, 605, 590, 610, 865, 250, 480, 545, 0, 
    655, 235, 585, 555, 750, 615, 625, 645, 775, 285, 515, 585, 190, 0, 
    480, 81, 435, 380, 575, 440, 455, 465, 600, 245, 345, 415, 295, 170, 0, 
    265, 480, 420, 235, 125, 125, 200, 165, 230, 475, 310, 205, 715, 650, 475, 0, 
    255, 440, 755, 235, 650, 370, 320, 350, 680, 150, 175, 265, 400, 435, 385, 485, 0, 
    450, 270, 625, 345, 660, 430, 420, 440, 690, 77, 310, 380, 180, 215, 190, 545, 225, 0, 
    170, 445, 750, 160, 495, 265, 220, 240, 600, 235, 125, 170, 485, 525, 405, 375, 87, 315, 0, 
    240, 290, 590, 140, 480, 255, 205, 220, 515, 150, 100, 170, 390, 425, 255, 395, 205, 220, 155, 0, 
    380, 140, 495, 280, 480, 340, 350, 370, 505, 185, 240, 310, 345, 280, 105, 380, 280, 165, 305, 150, 0,
]

gr17 = [
    0, 
    633, 0, 
    257, 390, 0, 
    91, 661, 228, 0, 
    412, 227, 169, 383, 0, 
    150, 488, 112, 120, 267, 0, 
    80, 572, 196, 77, 351, 63, 0, 
    134, 530, 154, 105, 309, 34, 29, 0, 
    259, 555, 372, 175, 338, 264, 232, 249, 0, 
    505, 289, 262, 476, 196, 360, 444, 402, 495, 0, 
    353, 282, 110, 324, 61, 208, 292, 250, 352, 154, 0, 
    324, 638, 437, 240, 421, 329, 297, 314, 95, 578, 435, 0, 
    70, 567, 191, 27, 346, 83, 47, 68, 189, 439, 287, 254, 0, 
    211, 466, 74, 182, 243, 105, 150, 108, 326, 336, 184, 391, 145, 0, 
    268, 420, 53, 239, 199, 123, 207, 165, 383, 240, 140, 448, 202, 57, 0, 
    246, 745, 472, 237, 528, 364, 332, 349, 202, 685, 542, 157, 289, 426, 483, 0, 
    121, 518, 142, 84, 297, 35, 29, 36, 236, 390, 238, 301, 55, 96, 153, 336, 0,
 ]

print("ejemploClase:", Grafo(ejemploClase).N, "ciudades")
print("gr17:", Grafo(gr17).N, "ciudades")
print("gr21:", Grafo(gr21).N, "ciudades")


ejemploClase: 6 ciudades
gr17: 12 ciudades
gr21: 21 ciudades


In [13]:
## TSP
print("______________H1")

p = SymmetricTSP(Grafo(gr17))

t0 = time.perf_counter()
sol = astar(p, h=p.h1)
t = time.perf_counter() - t0

acciones = path_actions(sol)
recorrido = [0] + [a for a in acciones if a != -1] + [0]

print("Solucion:", recorrido)
print("Coste:", sol.path_cost)
print("Tiempo:", t)
print("Nodos expandidos:", NODOS_EXPANDIDOS)
print("Estados reexpandidos:", ESTADOS_REEXPANDIDOS)
print("Estados alcanzados:", ESTADOS_ALCANZADOS)
print("Máximo frontier:", MAXIMO_FRONTIER)
print("Final frontier:", FINAL_FRONTIER)
print("Nodos dummy:", NODOS_DUMMY)
print("Actualizaciones en reached:", ACTUALIZACIONES_EN_REACHED)
print("Hijos descartados:", HIJOS_DESCARTADOS)

______________H1
Solucion: [0, 6, 7, 5, 2, 10, 9, 1, 4, 8, 11, 3, 0]
Coste: 1799
Tiempo: 0.09127119998447597
Nodos expandidos: 3371
Estados reexpandidos: 0
Estados alcanzados: 7176
Máximo frontier: 6422
Final frontier: 6417
Nodos dummy: 726
Actualizaciones en reached: 3339
Hijos descartados: 7175
